# GSE & GNMA Loans — 12-Month Performance Flags (May'24 & Jun'24)

**Purpose:** Build a loan-level training dataset for B2B pricing model — active GSE (FNMA/FHLMC) and GNMA loans as of May 2024 and June 2024, enriched with key loan characteristics and 12-month forward-looking performance flags.

| Parameter | Value |
|-----------|-------|
| Snapshot Months | May 2024 (`2024-05-31`) and Jun 2024 (`2024-06-30`) |
| Investor Filter | FNMA, FHLMC, GNMA (GSE + GNMA) |
| Observation Window | 12 months forward from each snapshot date |
| May'24 Window | 2024-05-31 → 2025-05-31 |
| Jun'24 Window | 2024-06-30 → 2025-06-30 |

**Output flags per loan × snapshot month:**
- `Payoff_Flag` — paid off anytime in the next 12 months (1/0)
- `DQ60_Flag` — became 60+ DPD anytime in the next 12 months (1/0)
- `Recapture_Flag` — recaptured by OCN anytime in the next 12 months (1/0)

## Setup & Parameters

In [49]:
import pyodbc
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# ============ SNAPSHOT MONTHS ============
# Edit this list to run for different months. Add more dates for multi-month pulls.
SNAPSHOT_MONTHS = ['2026-01-31']   # e.g. ['2024-05-31', '2024-06-30']

# ============ OBSERVATION WINDOW ============
# Change LOOKFORWARD_MONTHS to alter all windows (payoff, DQ, recapture checks).
# E.g. set to 1 for a 1-month forward view, 6 for 6-month, 12 for 1-year.
LOOKFORWARD_MONTHS = 1

# ============ DERIVED DATE PARAMETERS ============
# All parameters below are computed automatically — do NOT edit manually.

# Combined event pull window
COMBINED_WINDOW_START = SNAPSHOT_MONTHS[0]          # exclusive lower bound
COMBINED_WINDOW_END   = (
    pd.to_datetime(SNAPSHOT_MONTHS[-1]) + pd.offsets.MonthEnd(LOOKFORWARD_MONTHS)
).strftime('%Y-%m-%d')                               # inclusive upper bound

# SQL IN clause for snapshot months — used in Pull 1 and Pull 7
SNAPSHOT_IN_CLAUSE = ', '.join([f"'{d}'" for d in SNAPSHOT_MONTHS])

# PMMS date range — first day of first snapshot month to last day of last snapshot month
PMMS_START = pd.to_datetime(SNAPSHOT_MONTHS[0]).replace(day=1).strftime('%Y-%m-%d')
PMMS_END   = SNAPSHOT_MONTHS[-1]

# Next-month dates (always 1-month lookahead, for DQ_1M flag)
NEXT_MONTH_DATES = sorted(set([
    (pd.to_datetime(s) + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    for s in SNAPSHOT_MONTHS
]))
NEXT_MONTH_IN_CLAUSE = ', '.join([f"'{d}'" for d in NEXT_MONTH_DATES])

# ============ OTHER PARAMETERS ============
INVESTOR_FILTER = "('FNMA', 'FHLMC', 'GNMA')"
OUTPUT_DIR  = r'C:\Users\kanshash\OneDrive - Onity Group Inc\Desktop\Dev\B2B_cohortScore_update\cohorts_band'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'GSE_GNMA_Jan_2026_Performance_Flags.csv')
# ==========================================

print(f"Snapshot months    : {SNAPSHOT_MONTHS}")
print(f"Lookforward months : {LOOKFORWARD_MONTHS}")
print(f"Combined window    : {COMBINED_WINDOW_START}  →  {COMBINED_WINDOW_END}")
print(f"PMMS date range    : {PMMS_START}  →  {PMMS_END}")
print(f"Next-month dates   : {NEXT_MONTH_DATES}  (for DQ_1M flag)")
print(f"Output             : {OUTPUT_FILE}")

Snapshot months    : ['2026-01-31']
Lookforward months : 1
Combined window    : 2026-01-31  →  2026-02-28
PMMS date range    : 2026-01-01  →  2026-01-31
Next-month dates   : ['2026-02-28']  (for DQ_1M flag)
Output             : E:\Development\Shashi\B2B_Pricing_revamp\GSE_GNMA_Jan_2026_Performance_Flags.csv


## Database Connections

In [50]:
# ADM Connection (VMC2VMDBRADC01P)
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=VMC2VMDBRADC01P;"
    "DATABASE=ADM;"
    "Trusted_Connection=Yes;"
)
try:
    cn.cursor()
    print("✅ Reusing existing ADM connection")
except:
    cn = pyodbc.connect(conn_str)
    print("✅ Connected to ADM on VMC2VMDBRADC01P")

# DirectLendingRptDB Connection (IRVVMFLSDB01)
conn_str_dl = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=IRVVMFLSDB01;"
    "DATABASE=DirectLendingRptDB;"
    "Trusted_Connection=Yes;"
)
try:
    cn_dl.cursor()
    print("✅ Reusing existing DirectLending connection")
except:
    cn_dl = pyodbc.connect(conn_str_dl)
    print("✅ Connected to DirectLendingRptDB on IRVVMFLSDB01")

✅ Connected to ADM on VMC2VMDBRADC01P
✅ Connected to DirectLendingRptDB on IRVVMFLSDB01


## Pull 1 — Active GSE & GNMA Loan Attributes (Snapshot Months)

Pulls all active FNMA, FHLMC, and GNMA loans from `ADM.ODS.vwServLoanHist` for all months in `SNAPSHOT_MONTHS`.  
Key attributes: `InvestorCatgy`, `IntRate`, `PrinBal`, `FICOOrig`, `LoanAge`, `PropState`, `Channel`, `LoanPurpose`, `LoanType`, `AmortType`.

In [51]:
query_attr = f"""
SELECT
    s.LoanNum,
    s.AsOfDt,
    s.InvestorCatgy,
    CAST(s.IntRate * 100 AS DECIMAL(8,4))  AS IntRate,
    s.PrinBal,
    s.FICOOrig,
    DATEDIFF(MONTH, s.OrigDt, s.AsOfDt) AS LoanAge,
    s.PropState,
    s.Channel,
    s.LoanPurpose,
    s.LoanType,
    s.AmortType,
    s.LTVOrig,
    s.DelqStatusMBA,
    s.ArmFlag
FROM ADM.ODS.vwServLoanHist s
WHERE s.AsOfDt IN ({SNAPSHOT_IN_CLAUSE})
  AND s.ActiveFlag = 1
  AND s.InvestorCatgy IN {INVESTOR_FILTER}
"""
df_attr = pd.read_sql(query_attr, cn)
df_attr['LoanNum'] = df_attr['LoanNum'].str.strip()
df_attr['AsOfDt']  = pd.to_datetime(df_attr['AsOfDt'])

print(f"✅ Loan attributes: {df_attr.shape[0]:,} rows")
print(f"   IntRate sample: min={df_attr['IntRate'].min():.2f}%  max={df_attr['IntRate'].max():.2f}%  median={df_attr['IntRate'].median():.2f}%")
print(df_attr.groupby(['AsOfDt', 'InvestorCatgy'])['LoanNum'].count().unstack(fill_value=0).to_string())

✅ Loan attributes: 913,704 rows
   IntRate sample: min=0.00%  max=9.95%  median=3.62%
InvestorCatgy   FHLMC    FNMA    GNMA
AsOfDt                               
2026-01-31     367166  414482  132056


## Pull 2 — SellerName & AcqType (ServOrigLoan — SCD2)

`SellerName` and `AcqType` live in `ADM.ODS.ServOrigLoan` (SCD2 table).  
Use `WHERE EndDt = '9999-12-31'` to get the current record per loan.

In [52]:
query_orig = """
SELECT
    sol.LoanNum,
    sol.SellerName,
    sol.AcqType,
    sol.SubChannel
FROM ADM.ODS.ServOrigLoan sol
WHERE sol.EndDt = '9999-12-31'
"""
df_orig = pd.read_sql(query_orig, cn)
df_orig['LoanNum'] = df_orig['LoanNum'].str.strip()
print(f"✅ ServOrigLoan (SCD2 current): {df_orig.shape[0]:,} rows")
print(f"   AcqType distribution:\n{df_orig['AcqType'].value_counts().to_string()}")

✅ ServOrigLoan (SCD2 current): 3,177,726 rows
   AcqType distribution:
AcqType
Transfer      2240507
Purchased      668090
Originated     269128


## Pull 3 — Average PMMS by Snapshot Month

Fetches weekly 30-year fixed PMMS rates (`Series = 'MORTGAGE30US'`) from `ADM.Ext.MarketRates`  
and computes the **monthly average** for each month in `SNAPSHOT_MONTHS`.  
Date range is derived automatically: `PMMS_START` (1st of first snapshot month) → `PMMS_END` (last snapshot month-end).

In [53]:
query_pmms = f"""
SELECT
    CAST(EOMONTH(RateDt) AS DATE) AS AsOfDt,
    AVG(Rate)                     AS Avg_PMMS_Raw,
    COUNT(*)                      AS WeeklyObs
FROM ADM.Ext.MarketRates
WHERE Series = 'MORTGAGE30US'
  AND RateDt >= '{PMMS_START}'
  AND RateDt <= '{PMMS_END}'
GROUP BY EOMONTH(RateDt)
ORDER BY AsOfDt
"""
df_pmms = pd.read_sql(query_pmms, cn)
df_pmms['AsOfDt'] = pd.to_datetime(df_pmms['AsOfDt'])

# Rates are stored as fractions (0.0706 = 7.06%); convert to percentage to match IntRate scale
df_pmms['Market_Rate'] = (df_pmms['Avg_PMMS_Raw'] * 100).round(3)

print(f"✅ PMMS rates loaded: {df_pmms.shape[0]} months  ({PMMS_START} → {PMMS_END})")
print(df_pmms[['AsOfDt', 'Market_Rate', 'WeeklyObs']].to_string(index=False))

✅ PMMS rates loaded: 1 months  (2026-01-01 → 2026-01-31)
    AsOfDt  Market_Rate  WeeklyObs
2026-01-31        6.102          4


## Pull 4 — Payoff Events in Combined Observation Window (ADM)

Pulls all PIF (Paid-In-Full) events from `ADM.ODS.vwServLoanHist` within `COMBINED_WINDOW_START` → `COMBINED_WINDOW_END`.

The flag is assigned **per snapshot-month row**: `Payoff_Flag = 1` only if `ResolDt` falls within that row's `WindowEnd` (= `AsOfDt + LOOKFORWARD_MONTHS`).

In [54]:
# Pull earliest PIF date for each loan within the combined window
# ResolType = 'PIF' and ActiveFlag = 0 confirms the loan actually paid off
query_pif_adm = f"""
SELECT
    s.LoanNum,
    MIN(s.ResolDt) AS PaidOffDt_ADM
FROM ADM.ODS.vwServLoanHist s
WHERE s.ResolType = 'PIF'
  AND s.ResolDt  > '{COMBINED_WINDOW_START}'
  AND s.ResolDt  <= '{COMBINED_WINDOW_END}'
GROUP BY s.LoanNum
"""
df_pif_adm = pd.read_sql(query_pif_adm, cn)
df_pif_adm['LoanNum']      = df_pif_adm['LoanNum'].str.strip()
df_pif_adm['PaidOffDt_ADM'] = pd.to_datetime(df_pif_adm['PaidOffDt_ADM'])
print(f"✅ ADM PIF events in combined window: {df_pif_adm.shape[0]:,} loans")

✅ ADM PIF events in combined window: 8,212 loans


## Pull 5 — Payoff Events & Recapture Flag (DirectLendingRptDB)

Pulls payoff dates from `SAS_Model_Scores` and recapture confirmation from `RL_Dashboard_Details`  
(joined on `PriorLoanRecordationOriginalLoanNumber`). `fundssentdate IS NOT NULL` confirms recapture.

> Cross-server joins are not supported — pulled separately and merged in Python.

In [55]:
# 5a — PIF dates from SAS_Model_Scores (DirectLending)
query_pif_dl = f"""
SELECT DISTINCT
    a.loan_number                AS LoanNum,
    a.payment_in_full_date       AS PaidOffDt_DL
FROM DirectLendingRptDB.dbo.SAS_Model_Scores a WITH (NOLOCK)
WHERE a.payment_in_full_date > '{COMBINED_WINDOW_START}'
  AND a.payment_in_full_date <= '{COMBINED_WINDOW_END}'
"""
df_pif_dl = pd.read_sql(query_pif_dl, cn_dl)
df_pif_dl['LoanNum']      = df_pif_dl['LoanNum'].str.strip().str.zfill(10)
df_pif_dl['PaidOffDt_DL'] = pd.to_datetime(df_pif_dl['PaidOffDt_DL'])
# Keep earliest PIF date per loan
df_pif_dl = df_pif_dl.sort_values('PaidOffDt_DL').drop_duplicates('LoanNum', keep='first')
print(f"✅ DirectLending PIF in window: {df_pif_dl.shape[0]:,} loans")

# 5b — Recapture confirmation + RecaptureBalance from RL_Dashboard_Details
# borrowerrequestedloanamount = balance of the new loan that recaptured the payoff
query_recap = f"""
SELECT
    a.loan_number                              AS LoanNum,
    MIN(b.fundssentdate)                       AS RecaptureDt,
    MAX(b.borrowerrequestedloanamount)         AS RecaptureBalance
FROM DirectLendingRptDB.dbo.SAS_Model_Scores a WITH (NOLOCK)
JOIN DirectLendingRptDB.dbo.RL_Dashboard_Details b WITH (NOLOCK)
    ON a.loan_number = b.PriorLoanRecordationOriginalLoanNumber
WHERE b.fundssentdate IS NOT NULL
  AND a.payment_in_full_date > '{COMBINED_WINDOW_START}'
  AND a.payment_in_full_date <= '{COMBINED_WINDOW_END}'
GROUP BY a.loan_number
"""
df_recap = pd.read_sql(query_recap, cn_dl)
df_recap['LoanNum']          = df_recap['LoanNum'].str.strip().str.zfill(10)
df_recap['RecaptureDt']      = pd.to_datetime(df_recap['RecaptureDt'])
df_recap['RecaptureBalance'] = pd.to_numeric(df_recap['RecaptureBalance'], errors='coerce')
print(f"✅ Recaptured loans in window: {df_recap.shape[0]:,} loans")

✅ DirectLending PIF in window: 0 loans
✅ Recaptured loans in window: 0 loans


## Pull 6 — 60+ DPD Events in Combined Observation Window

Finds the **earliest month** each loan hit 60 or more days past due within the combined window.  
`DelqStatusMBA` buckets included: `60-89`, `90-119`, `120-149`, `150-179`, `180+`, `BK`, `FC`, `REO`.  
Flag is then applied per snapshot month's 12-month window in Python.

In [56]:
query_dq = f"""
SELECT
    s.LoanNum,
    MIN(s.AsOfDt)    AS First_DQ60_Dt,
    MAX(s.DelqStatusMBA) AS Worst_DQ_Status
FROM ADM.ODS.vwServLoanHist s
WHERE s.AsOfDt  > '{COMBINED_WINDOW_START}'
  AND s.AsOfDt  <= '{COMBINED_WINDOW_END}'
  AND s.DelqStatusMBA IN ('60-89', '90-119', '120-149', '150-179', '180+', 'BK', 'FC', 'REO')
GROUP BY s.LoanNum
"""
df_dq = pd.read_sql(query_dq, cn)
df_dq['LoanNum']       = df_dq['LoanNum'].str.strip()
df_dq['First_DQ60_Dt'] = pd.to_datetime(df_dq['First_DQ60_Dt'])
print(f"✅ Loans with 60+ DPD in combined window: {df_dq.shape[0]:,}")

✅ Loans with 60+ DPD in combined window: 70,069


## Pull 7 — Next-Month DQ Status (for DQ_1M Flag)

Pulls `DelqStatusMBA` for the **month immediately after** each snapshot date.  
Always a 1-month lookahead — independent of `LOOKFORWARD_MONTHS`.  
Next-month dates are derived automatically as `NEXT_MONTH_DATES` in the parameters cell.

In [57]:
# Pull DelqStatusMBA for the next-month snapshot dates
query_dq_1m = f"""
SELECT
    s.LoanNum,
    s.AsOfDt     AS NextMonthDt,
    s.DelqStatusMBA AS NextMonthDelqStatus
FROM ADM.ODS.vwServLoanHist s
WHERE s.AsOfDt IN ({NEXT_MONTH_IN_CLAUSE})
"""
df_dq_1m = pd.read_sql(query_dq_1m, cn)
df_dq_1m['LoanNum']     = df_dq_1m['LoanNum'].str.strip()
df_dq_1m['NextMonthDt'] = pd.to_datetime(df_dq_1m['NextMonthDt'])

print(f"✅ Next-month DQ lookup: {df_dq_1m.shape[0]:,} rows")
print(df_dq_1m.groupby('NextMonthDt')['LoanNum'].count().to_string())

✅ Next-month DQ lookup: 1,394,937 rows
NextMonthDt
2026-02-28    1394937


## Build Final Dataset

Merge all pulls together and compute the three forward-looking performance flags.

**Window logic (per row):**
- `window_end` = `AsOfDt` + 12 months (e.g., 2024-05-31 → 2025-05-31)
- `Payoff_Flag = 1` if earliest PIF date (ADM or DL) ≤ `window_end`
- `DQ60_Flag = 1` if earliest 60+ DPD occurrence ≤ `window_end`
- `Recapture_Flag = 1` if `RecaptureDt` ≤ `window_end` (recapture requires PIF first)

In [58]:
# --- Step 1: Merge SellerName / AcqType onto base loan attributes ---
df = df_attr.merge(df_orig[['LoanNum', 'SellerName', 'AcqType', 'SubChannel']],
                   on='LoanNum', how='left')

# --- Step 2: Merge PMMS (join on AsOfDt = month-end date) ---
df = df.merge(df_pmms[['AsOfDt', 'Market_Rate']], on='AsOfDt', how='left')

# --- Step 3: Merge event tables (all are LoanNum-level, not per-snapshot) ---
df = df.merge(df_pif_adm[['LoanNum', 'PaidOffDt_ADM']], on='LoanNum', how='left')
df = df.merge(df_pif_dl[['LoanNum', 'PaidOffDt_DL']],   on='LoanNum', how='left')
df = df.merge(df_recap[['LoanNum', 'RecaptureDt', 'RecaptureBalance']], on='LoanNum', how='left')
df = df.merge(df_dq[['LoanNum', 'First_DQ60_Dt', 'Worst_DQ_Status']],
              on='LoanNum', how='left')

# --- Step 4: Merge next-month DQ for DQ_1M flag ---
# Build a join key: each row needs LoanNum + the next-month date for its snapshot month
df['NextMonthDt'] = df['AsOfDt'] + pd.offsets.MonthEnd(1)
df = df.merge(
    df_dq_1m[['LoanNum', 'NextMonthDt', 'NextMonthDelqStatus']],
    on=['LoanNum', 'NextMonthDt'], how='left'
)

# --- Step 5: Compute observation window_end (driven by LOOKFORWARD_MONTHS parameter) ---
df['WindowEnd'] = df['AsOfDt'] + pd.offsets.MonthEnd(LOOKFORWARD_MONTHS)

# --- Step 6: Payoff Flag ---
# Use the earlier of ADM and DL PIF date as the definitive payoff date
df['FirstPIFDt'] = df[['PaidOffDt_ADM', 'PaidOffDt_DL']].min(axis=1)
df['Payoff_Flag'] = (
    df['FirstPIFDt'].notna() &
    (df['FirstPIFDt'] > df['AsOfDt']) &
    (df['FirstPIFDt'] <= df['WindowEnd'])
).astype(int)

# --- Step 7: BeginPrinBal & EndPrinBal ---
# BeginPrinBal = PrinBal at snapshot date
# EndPrinBal   = 0 if loan paid off in the NEXT calendar month; otherwise = BeginPrinBal
df['BeginPrinBal'] = df['PrinBal']
next_month_end = df['AsOfDt'] + pd.offsets.MonthEnd(1)
paid_off_next_month = (
    df['FirstPIFDt'].notna() &
    (df['FirstPIFDt'] > df['AsOfDt']) &
    (df['FirstPIFDt'] <= next_month_end)
)
df['EndPrinBal'] = np.where(paid_off_next_month, 0.0, df['BeginPrinBal'])

# --- Step 8: DQ60 Flag ---
df['DQ60_Flag'] = (
    df['First_DQ60_Dt'].notna() &
    (df['First_DQ60_Dt'] > df['AsOfDt']) &
    (df['First_DQ60_Dt'] <= df['WindowEnd'])
).astype(int)

# --- Step 9: DQ_1M Flag — did DQ status worsen in the immediately next month? ---
DQ_ORDER = {
    '0-29': 0, '30-59': 1, '60-89': 2, '90-119': 3,
    '120-149': 4, '150-179': 5, '180+': 6, 'BK': 7, 'FC': 8, 'REO': 9
}
curr_rank = df['DelqStatusMBA'].map(DQ_ORDER).fillna(0)
next_rank = df['NextMonthDelqStatus'].map(DQ_ORDER).fillna(-1)
# Flag = 1 only when next month status is strictly worse AND next month data exists
df['DQ_1M'] = (
    df['NextMonthDelqStatus'].notna() &
    (next_rank > curr_rank)
).astype(int)

# --- Step 10: Recapture Flag ---
# Recapture requires: loan paid off in window AND recapture date in window
df['Recapture_Flag'] = (
    (df['Payoff_Flag'] == 1) &
    df['RecaptureDt'].notna() &
    (df['RecaptureDt'] > df['AsOfDt']) &
    (df['RecaptureDt'] <= df['WindowEnd'])
).astype(int)

# --- Step 11: Rate Spread (RateSpread = IntRate - Market_Rate; positive = in the money) ---
df['RateSpread'] = (df['IntRate'] - df['Market_Rate']).round(3)

print(f"✅ Final dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"   BeginPrinBal: ${df['BeginPrinBal'].sum():,.0f}")
print(f"   EndPrinBal=0 (paid off next month): {(df['EndPrinBal'] == 0).sum():,}")
print(f"   DQ_1M degraded: {df['DQ_1M'].sum():,} ({df['DQ_1M'].mean()*100:.2f}%)")
df.head(3)

✅ Final dataset: 913,704 rows x 36 columns
   BeginPrinBal: $243,632,355,422
   EndPrinBal=0 (paid off next month): 5,845
   DQ_1M degraded: 12,773 (1.40%)


,LoanNum,AsOfDt,InvestorCatgy,IntRate,PrinBal,FICOOrig,LoanAge,PropState,Channel,LoanPurpose,...,NextMonthDelqStatus,WindowEnd,FirstPIFDt,Payoff_Flag,BeginPrinBal,EndPrinBal,DQ60_Flag,DQ_1M,Recapture_Flag,RateSpread
0,7920059602,2026-01-31,FHLMC,6.980,553621.82,710.0,4,OR,SUBSERVICING CORRESPONDENT,PURCHASE,...,0-29,2026-02-28,NaT,0,553621.82,553621.82,0,0,0,0.878
1,7912962292,2026-01-31,FNMA,4.625,69475.45,NaN,201,CA,MSR - BULK,REFIANCE/EQUITY TAKEOUT,...,0-29,2026-02-28,NaT,0,69475.45,69475.45,0,0,0,-1.477
2,7913505967,2026-01-31,FNMA,6.980,170075.81,785.0,6,OR,SUBSERVICING CORRESPONDENT,PURCHASE,...,0-29,2026-02-28,NaT,0,170075.81,170075.81,0,0,0,0.878


## Data Quality & Summary

In [59]:
total = len(df)
print(f"{'='*65}")
print(f"PERFORMANCE FLAGS SUMMARY  ({total:,} loan×month rows)")
print(f"Lookforward window: {LOOKFORWARD_MONTHS} month(s)")
print(f"{'='*65}")

# Per-snapshot breakdown
for snap_dt in sorted(df['AsOfDt'].unique()):
    snap_str = pd.Timestamp(snap_dt).strftime('%Y-%m-%d')
    sub = df[df['AsOfDt'] == snap_dt]
    n   = len(sub)
    print(f"\n  Snapshot: {snap_str}   |   Loans: {n:,}")
    print(f"    InvestorCatgy    : {sub['InvestorCatgy'].value_counts().to_dict()}")
    print(f"    Payoff_Flag      : {sub['Payoff_Flag'].sum():,} ({sub['Payoff_Flag'].mean()*100:.1f}%)")
    print(f"    DQ60_Flag        : {sub['DQ60_Flag'].sum():,} ({sub['DQ60_Flag'].mean()*100:.1f}%)")
    print(f"    DQ_1M Flag       : {sub['DQ_1M'].sum():,} ({sub['DQ_1M'].mean()*100:.2f}%)")
    print(f"    Recapture_Flag   : {sub['Recapture_Flag'].sum():,} ({sub['Recapture_Flag'].mean()*100:.2f}%)")
    print(f"    Avg PMMS         : {sub['Market_Rate'].mean():.2f}%")
    print(f"    Avg IntRate      : {sub['IntRate'].mean():.2f}%")
    print(f"    Avg RateSpread   : {sub['RateSpread'].mean():.2f}%")
    print(f"    Total UPB (Begin): ${sub['BeginPrinBal'].sum():,.0f}")
    print(f"    EndPrinBal=0     : {(sub['EndPrinBal']==0).sum():,} (paid off in next month)")

print(f"\n{'='*65}")
print("\nMissing data audit (key columns):")
key_cols = ['InvestorCatgy', 'IntRate', 'BeginPrinBal', 'EndPrinBal',
            'FICOOrig', 'LoanAge', 'LoanPurpose',
            'PropState', 'Channel', 'AcqType', 'SellerName', 'Market_Rate',
            'Payoff_Flag', 'DQ60_Flag', 'DQ_1M', 'Recapture_Flag']
missing = pd.DataFrame({
    'Column': key_cols,
    'Missing': [df[c].isnull().sum() for c in key_cols],
    'Missing%': [(df[c].isnull().sum() / total * 100).round(1) for c in key_cols]
})
print(missing[missing['Missing'] > 0].to_string(index=False) or "  → No missing values in key columns")

PERFORMANCE FLAGS SUMMARY  (913,704 loan×month rows)
Lookforward window: 1 month(s)

  Snapshot: 2026-01-31   |   Loans: 913,704
    InvestorCatgy    : {'FNMA': 414482, 'FHLMC': 367166, 'GNMA': 132056}
    Payoff_Flag      : 5,845 (0.6%)
    DQ60_Flag        : 14,262 (1.6%)
    DQ_1M Flag       : 12,773 (1.40%)
    Recapture_Flag   : 0 (0.00%)
    Avg PMMS         : 6.10%
    Avg IntRate      : 4.27%
    Avg RateSpread   : -1.83%
    Total UPB (Begin): $243,632,355,422
    EndPrinBal=0     : 5,845 (paid off in next month)


Missing data audit (key columns):
     Column  Missing  Missing%
    IntRate       26       0.0
   FICOOrig   145450      15.9
LoanPurpose      122       0.0
 SellerName    25555       2.8


## Flag Distribution by InvestorCatgy & Snapshot Month

In [60]:
summary = (
    df.groupby(['AsOfDt', 'InvestorCatgy'])
    .agg(
        LoanCount      = ('LoanNum',        'count'),
        UPB_M          = ('BeginPrinBal',   lambda x: x.sum() / 1e6),
        Payoff_Rate    = ('Payoff_Flag',    'mean'),
        DQ60_Rate      = ('DQ60_Flag',      'mean'),
        DQ_1M_Rate     = ('DQ_1M',          'mean'),
        Recapture_Rate = ('Recapture_Flag', 'mean'),
        Avg_IntRate    = ('IntRate',        'mean'),
        Market_Rate    = ('Market_Rate',    'mean'),
        Avg_RateSpread = ('RateSpread',     'mean'),
    )
    .reset_index()
)
summary['AsOfDt']        = summary['AsOfDt'].dt.strftime('%Y-%m-%d')
summary['UPB_M']         = summary['UPB_M'].map('${:,.0f}M'.format)
summary['Payoff_Rate']   = summary['Payoff_Rate'].map('{:.1%}'.format)
summary['DQ60_Rate']     = summary['DQ60_Rate'].map('{:.1%}'.format)
summary['DQ_1M_Rate']    = summary['DQ_1M_Rate'].map('{:.2%}'.format)
summary['Recapture_Rate']= summary['Recapture_Rate'].map('{:.2%}'.format)
summary['Avg_IntRate']   = summary['Avg_IntRate'].map('{:.2f}%'.format)
summary['Market_Rate']   = summary['Market_Rate'].map('{:.2f}%'.format)
summary['Avg_RateSpread']= summary['Avg_RateSpread'].map('{:+.2f}%'.format)
summary['LoanCount']     = summary['LoanCount'].map('{:,}'.format)

print(summary.to_string(index=False))

    AsOfDt InvestorCatgy LoanCount     UPB_M Payoff_Rate DQ60_Rate DQ_1M_Rate Recapture_Rate Avg_IntRate Market_Rate Avg_RateSpread
2026-01-31         FHLMC   367,166 $100,403M        0.7%      0.6%      0.77%          0.00%       4.26%       6.10%         -1.84%
2026-01-31          FNMA   414,482 $108,065M        0.6%      1.1%      1.07%          0.00%       4.04%       6.10%         -2.06%
2026-01-31          GNMA   132,056  $35,164M        0.7%      5.7%      4.16%          0.00%       5.01%       6.10%         -1.09%


## Key Takeaways

In [61]:
print("KEY TAKEAWAYS")
print("="*65)

for snap_dt in sorted(df['AsOfDt'].unique()):
    snap_str = pd.Timestamp(snap_dt).strftime('%b\'%y')
    sub  = df[df['AsOfDt'] == snap_dt]
    n    = len(sub)
    pif  = sub['Payoff_Flag'].mean() * 100
    dq60 = sub['DQ60_Flag'].mean() * 100
    dq1m = sub['DQ_1M'].mean() * 100
    rec  = sub['Recapture_Flag'].mean() * 100
    pmms = sub['Market_Rate'].mean()

    print(f"\n  {snap_str} Snapshot ({n:,} active GSE/GNMA loans)")
    print(f"  • {LOOKFORWARD_MONTHS}-month Payoff Rate   : {pif:.1f}%  "
          f"(Avg RateSpread: {sub['RateSpread'].mean():+.2f}% vs PMMS {pmms:.2f}%)")
    print(f"  • {LOOKFORWARD_MONTHS}-month 60+ DPD Rate  : {dq60:.1f}%")
    print(f"  • 1-month DQ Degradation  : {dq1m:.2f}%")
    print(f"  • {LOOKFORWARD_MONTHS}-month Recapture Rate : {rec:.2f}%  "
          f"(of payoffs recaptured: "
          f"{sub[sub['Payoff_Flag']==1]['Recapture_Flag'].mean()*100:.1f}%)")

print(f"\n  {'─'*62}")
print(f"  Output saved: {OUTPUT_FILE}")

KEY TAKEAWAYS

  Jan'26 Snapshot (913,704 active GSE/GNMA loans)
  • 1-month Payoff Rate   : 0.6%  (Avg RateSpread: -1.83% vs PMMS 6.10%)
  • 1-month 60+ DPD Rate  : 1.6%
  • 1-month DQ Degradation  : 1.40%
  • 1-month Recapture Rate : 0.00%  (of payoffs recaptured: 0.0%)

  ──────────────────────────────────────────────────────────────
  Output saved: E:\Development\Shashi\B2B_Pricing_revamp\GSE_GNMA_Jan_2026_Performance_Flags.csv


## Save Output

In [62]:
# Column order for final output
output_cols = [
    # Identity
    'LoanNum', 'AsOfDt',
    # Loan characteristics
    'InvestorCatgy', 'IntRate', 'BeginPrinBal', 'EndPrinBal', 'FICOOrig', 'LoanAge',
    'LTVOrig', 'AmortType', 'ArmFlag', 'LoanType', 'LoanPurpose',
    'PropState', 'Channel', 'SubChannel', 'AcqType', 'SellerName',
    'DelqStatusMBA',
    # Market rates
    'Market_Rate', 'RateSpread',
    # Performance flags & supporting dates
    'Payoff_Flag', 'FirstPIFDt', 'PaidOffDt_ADM', 'PaidOffDt_DL',
    'DQ60_Flag', 'First_DQ60_Dt', 'Worst_DQ_Status',
    'DQ_1M', 'NextMonthDelqStatus',
    'Recapture_Flag', 'RecaptureDt', 'RecaptureBalance',
    # Window reference
    'WindowEnd',
]
# Keep only columns that exist (safety check)
output_cols = [c for c in output_cols if c in df.columns]
df_out = df[output_cols].copy()
df_out['AsOfDt']    = df_out['AsOfDt'].dt.strftime('%Y-%m-%d')
df_out['WindowEnd'] = df_out['WindowEnd'].dt.strftime('%Y-%m-%d')

df_out.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Saved: {OUTPUT_FILE}")
print(f"   {df_out.shape[0]:,} rows × {df_out.shape[1]} columns")

✅ Saved: E:\Development\Shashi\B2B_Pricing_revamp\GSE_GNMA_Jan_2026_Performance_Flags.csv
   913,704 rows × 34 columns


## Close Connections

In [63]:
try:
    cn.close()
    print("✅ ADM connection closed")
except:
    pass

try:
    cn_dl.close()
    print("✅ DirectLending connection closed")
except:
    pass

✅ ADM connection closed
✅ DirectLending connection closed
